# Define isoform specific peptides (unique in digestion output)
In this notebook we will define unique peptides for the isoforms that had a unique peptide in the digestion output (this is roughly 2/3 of all isoforms). We will need to add a C- and N-terminus to the fragments that don't have one or either and put the results into fasta format.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
from Bio import SeqIO
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/03_Isoform_Specific_Peptides"):
    print("WARNING: The working directory is not set to the '03_Isoform_Specific_Peptides'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '03_Isoform_Specific_Peptides' folder (\"{cwd}\").")

In [ ]:
# Data directories
digestion_dir = "../02_In-Silico_Digestion/data/"
unique_dir = "../01_UniProt/data/unique/"

## Read in Dataframe

In [ ]:
unique_isoform_peptides = pd.read_csv(os.path.join(digestion_dir, 'unique_isoform_peptides.csv'))
sequential_isoform_peptides = pd.read_csv(os.path.join(digestion_dir, 'sequential_isoform_peptides.csv'))
sequential_isoform_peptides

# Check how many peptides are tryptic
A "tryptic" peptide is defined by how the enzyme Trypsin cleaves a protein. Trypsin is very specific: it cuts the protein chain after every Lysine (K) and Arginine (R), but usually skips them if they are followed by a Proline (P).

The non-tryptic ones are probably at the protein C-terminus: The peptide was the very last part of the protein. These are "naturally" non-tryptic at the end but are still valid for MaxQuant.

In [ ]:
sequential_isoform_peptides['Is_Tryptic'] = sequential_isoform_peptides['Unique_Peptide'].str.endswith(('K', 'R'), na=False)

# Show only the rows where a unique peptide was found but it's NOT tryptic
non_tryptic_markers = sequential_isoform_peptides[
    (sequential_isoform_peptides['Status'].str.contains("Unique Marker Found", na=False)) &
    (sequential_isoform_peptides['Is_Tryptic'] == False)
].copy()
print(len(non_tryptic_markers))

In [ ]:
#check if the non-tryptic peptides are C-termini by comparing to full sequence
full_proteome_unique = pd.read_csv(os.path.join(unique_dir, 'full_proteome_unique.csv'))

# Create a dictionary for fast sequence lookups
sequence_dict = dict(zip(full_proteome_unique['ID'], full_proteome_unique['seq']))

def check_c_terminus(row, seq_dict):
    iso_id = row['Isoform_ID']
    peptide = row['Unique_Peptide']
    
    # Handle NaNs or missing IDs
    if not isinstance(peptide, str) or iso_id not in seq_dict:
        return False
    
    full_protein_seq = seq_dict[iso_id]
    
    # Check if the full sequence ends with the peptide sequence
    return full_protein_seq.endswith(peptide)


sequential_isoform_peptides['Is_C_Terminus'] = sequential_isoform_peptides.apply(
    lambda row: check_c_terminus(row, sequence_dict), axis=1
)

In [ ]:
def get_validation_status(row):
    if row['Is_Tryptic']:
        return "Standard Tryptic"
    if row['Is_C_Terminus']:
        return "C-Terminal (Valid)"


sequential_isoform_peptides['Validation'] = sequential_isoform_peptides.apply(get_validation_status, axis=1)

# Let's see how many valid peptides we have!
print(sequential_isoform_peptides['Validation'].value_counts())

13950+2075=16’025 which is the number of isoform specific peptides found. This means that all of these are either standard tryptic or C-terminal peptides and will be handled normally by MaxQuant. 

# Create a fasta file for first unique peptides

In [ ]:
fasta_output_path = "data/unique_isoform_peptides_1.fasta"

with open(fasta_output_path, "w") as f:
    # Only iterate over rows where a unique peptide was actually found
    marker_df = sequential_isoform_peptides[sequential_isoform_peptides['Status'].str.contains("Unique Marker Found", na=False)]
    
    for _, row in marker_df.iterrows():
        iso_id = row['Isoform_ID']
        peptide_seq = row['Unique_Peptide']
        
        # Create a standardized UniProt-style header
        # Format: >db|UniqueIdentifier|EntryName
        header = f">marker|{iso_id}_pep|UNIQUE_PEPTIDE_{iso_id}"
        
        f.write(f"{header}\n")
        f.write(f"{peptide_seq}\n")

print(f"Successfully created FASTA with {len(marker_df)} entries.")